# Serverless LLMs and Agentic AI with Modal – Lesson 3
## Custom Images: Packages, Shell Commands, and Shipping Local Assets

In Lesson 1–2 you learned how to run functions remotely and tune scaling.  
In Lesson 3 we focus on **the environment your code runs in**.

By the end of this notebook, you’ll be able to:

- Build a **custom Modal image** (your own container environment).
- Install **system packages** (via `apt_install`) and **Python packages** (via `pip_install`).
- Run **shell commands during the image build** (`run_commands`) to fetch/build resources.
- Ship **local assets** (templates, config files) into the container (`add_local_dir`, `add_local_file`).
- Understand a key workflow trick: **keep `add_local_*` steps last** to avoid slow rebuilds.

We’ll create a small *“Report Generator Service”*:
- During image build, the container downloads a public-domain text file (a book) into `/data/`.
- At runtime, a Modal function reads that text, computes simple stats + top words, and returns a **Markdown report**.
- Another function renders an **HTML report** using a Jinja2 template shipped from your notebook’s local `assets/` folder.

> This is a pattern you’ll reuse for LLM apps:
> - Install model/runtime deps in the image
> - Fetch model artifacts during build
> - Ship prompts/templates/config as local assets
> - Return portable data types (strings/dicts) back to the caller


In [ ]:
# =====================================
# Step 0 – Install and check Modal
# =====================================

!pip install modal --quiet
!which modal
!modal --version

print("✅ Modal installed.")

## Step 1 – Verify authentication

If `modal token list` fails:
1) Create a token in the Modal dashboard (Profile → Tokens)  
2) Run in a terminal:
```bash
modal token set --token-id <YOUR_TOKEN_ID> --token-secret <YOUR_TOKEN_SECRET>
```



In [ ]:
TOKEN_ID = ""        # <-- paste from Modal dashboard
TOKEN_SECRET = ""  # <-- paste from Modal dashboard



if "YOUR_TOKEN_ID_HERE" in TOKEN_ID or "YOUR_TOKEN_SECRET_HERE" in TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

# Call the Modal CLI to store the token
!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

In [ ]:
!modal token -h || echo "⚠️ Auth error — run: modal token set --token-id ... --token-secret ..."


## Step 2 – Create local assets

We’ll make a local folder called `assets/` containing:

- `stopwords.txt` – words we ignore in our word-frequency summary
- `report_template.html` – a Jinja2 template for rendering HTML

These files will be shipped into the container image via `image.add_local_dir(...)`.

> Tip: In real projects, `assets/` might contain prompt templates, JSON configs, UI templates, etc.


In [ ]:
# Create assets directory and a couple of files
import os, textwrap, pathlib

pathlib.Path("assets").mkdir(exist_ok=True)

# A small stopwords list (you can expand it)
stopwords = [
    "the","and","to","of","a","in","that","it","is","was","i","for","on","you","with",
    "as","at","by","but","be","this","have","not","are","from","or","they","an","his","her",
]
with open("assets/stopwords.txt", "w") as f:
    f.write("\n".join(stopwords) + "\n")

# A simple HTML template (Jinja2)
html_template = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>{{ title }}</title>
  <style>
    body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 24px; }
    .card { border: 1px solid #e5e7eb; border-radius: 14px; padding: 16px; max-width: 900px; }
    code { background: #f3f4f6; padding: 2px 6px; border-radius: 8px; }
    table { border-collapse: collapse; width: 100%; }
    th, td { border-bottom: 1px solid #eee; padding: 8px; text-align: left; }
  </style>
</head>
<body>
  <div class="card">
    <h1>{{ title }}</h1>
    <p><b>Source:</b> <code>{{ source_path }}</code></p>
    <p><b>Generated at:</b> {{ generated_at }}</p>

    <h2>Stats</h2>
    <ul>
      <li>Total characters: {{ stats.total_chars }}</li>
      <li>Total words: {{ stats.total_words }}</li>
      <li>Unique words (after filtering): {{ stats.unique_words }}</li>
    </ul>

    <h2>Top Words</h2>
    <table>
      <thead><tr><th>Word</th><th>Count</th></tr></thead>
      <tbody>
        {% for row in top_words %}
          <tr><td>{{ row.word }}</td><td>{{ row.count }}</td></tr>
        {% endfor %}
      </tbody>
    </table>
  </div>
</body>
</html>
"""
with open("assets/report_template.html", "w") as f:
    f.write(textwrap.dedent(html_template).strip() + "\n")

print("✅ Created assets/stopwords.txt and assets/report_template.html")

## Step 3 – Write the Modal app file

We’ll create `lesson3_images.py` with:

### Image build steps
- `modal.Image.debian_slim(...)` – base OS + Python
- `.apt_install("curl")` – system package
- `.pip_install(...)` – Python dependencies (only in the container)
- `.run_commands(...)` – download a text file into `/data/book.txt`
- `.add_local_dir("assets", "/assets")` – ship our local assets
- `.add_local_file(__file__, "/src/lesson3_images.py")` – ship this file itself (helpful for debugging)

### Functions
- `make_markdown_report()` → returns Markdown (portable; no local deps needed)
- `render_html_report()` → returns HTML string rendered from the Jinja2 template

> Note: We return strings (Markdown/HTML). If you return a complex object (like a pandas DataFrame),
> your *local* environment would need that library too to deserialize it.


In [ ]:
%%writefile lesson3_images.py
import re
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import List

import modal

app = modal.App("lesson3-custom-images-report-generator")

# ------------------------------------------------------------
# Custom Image Definition
# ------------------------------------------------------------
# New project context:
#   A tiny “Report Generator Service”
#   - Build-time: download a public-domain book into /data/book.txt
#   - Runtime: compute top words + stats and return Markdown/HTML
#
# Key workflow tip:
#   Put add_local_* steps LAST so changing local files doesn’t force
#   rebuilding everything from scratch.
# ------------------------------------------------------------

BOOK_URL = "https://www.gutenberg.org/cache/epub/11/pg11.txt"  # Alice in Wonderland
BOOK_PATH = "/data/book.txt"

image = (
    modal.Image.debian_slim(python_version="3.11")
    .apt_install("curl")
    .pip_install(
        "jinja2==3.1.4",
        "regex==2024.7.24",
        "orjson==3.10.7",
    )
    .run_commands(
        "mkdir -p /data",
        f"curl -L -s -o {BOOK_PATH} {BOOK_URL}",
        f"python -c \"print('Downloaded bytes:', len(open('{BOOK_PATH}','rb').read()))\"",
    )
    # Keep local additions at the end for faster rebuild cycles
    .add_local_dir("assets", remote_path="/assets")
    .add_local_file(__file__, remote_path="/src/lesson3_images.py")
)


@dataclass
class TopWord:
    word: str
    count: int


def _load_stopwords() -> set:
    """Load stopwords shipped into the container under /assets/."""
    with open("/assets/stopwords.txt", "r", encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}


def _tokenize(text: str) -> List[str]:
    """Simple tokenizer: keep letters/apostrophes, drop punctuation/numbers."""
    return re.findall(r"[A-Za-z']+", text.lower())


@app.function(image=image)
def make_markdown_report(top_k: int = 20) -> str:
    """
    Compute stats + top words and return a Markdown report (string).

    Returning a string is portable:
    - The caller (your local machine / Colab) doesn’t need extra deps
      to deserialize complex objects.
    """
    stop = _load_stopwords()

    with open(BOOK_PATH, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    raw_tokens = _tokenize(text)
    tokens = [t for t in raw_tokens if t not in stop and len(t) > 1]
    counts = Counter(tokens).most_common(top_k)

    total_chars = len(text)
    total_words = len(raw_tokens)
    unique_words = len(set(tokens))

    lines = []
    lines.append("# Book Report (Markdown)\n")
    lines.append(f"**Source file:** `{BOOK_PATH}`\n\n")
    lines.append("## Stats\n")
    lines.append(f"- Total characters: **{total_chars}**\n")
    lines.append(f"- Total words (raw): **{total_words}**\n")
    lines.append(f"- Unique words (after filtering): **{unique_words}**\n\n")
    lines.append("## Top Words\n\n")
    lines.append("| Word | Count |\n|---|---:|")
    for w, c in counts:
        lines.append(f"\n| {w} | {c} |")

    return "".join(lines) + "\n"


@app.function(image=image)
def render_html_report(top_k: int = 20) -> str:
    """
    Render an HTML report using a Jinja2 template shipped via add_local_dir.
    """
    from jinja2 import Template

    stop = _load_stopwords()

    with open(BOOK_PATH, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    raw_tokens = _tokenize(text)
    tokens = [t for t in raw_tokens if t not in stop and len(t) > 1]
    top = [TopWord(word=w, count=c) for (w, c) in Counter(tokens).most_common(top_k)]

    stats = {
        "total_chars": len(text),
        "total_words": len(raw_tokens),
        "unique_words": len(set(tokens)),
    }

    with open("/assets/report_template.html", "r", encoding="utf-8") as f:
        tmpl = Template(f.read())

    html = tmpl.render(
        title="Lesson 3 – Custom Image Report Generator",
        source_path=BOOK_PATH,
        generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        stats=stats,
        top_words=top,
    )
    return html


@app.local_entrypoint()
def lesson3_main(top_k: int = 20):
    """
    Entrypoint for:
      modal run lesson3_images.py --top-k 20
    """
    print("\n=== Markdown report (first 40 lines) ===\n")
    md = make_markdown_report.remote(top_k=top_k)
    print("\n".join(md.splitlines()[:40]))

    print("\n=== HTML report (first 30 lines) ===\n")
    html = render_html_report.remote(top_k=top_k)
    print("\n".join(html.splitlines()[:30]))

    print("\n✅ Done. Open the Modal dashboard to inspect the image build + logs.")


## Step 4 – Run it on Modal

This will trigger an **image build** the first time (you’ll see logs about installing packages and downloading the book).  
Subsequent runs should be faster due to **image layer caching**.


In [ ]:
!modal run lesson3_images.py --top-k 20

## Step 5 – Debugging inside the container

You can open a shell inside the container for a specific function.

Try (in a terminal):
```bash
modal shell lesson3_images.py::make_markdown_report
```
Inside the shell, explore:
- `ls /data`
- `ls /assets`
- `head -n 5 /data/book.txt`
- `cat /src/lesson3_images.py | head`

This is extremely useful when an image build worked but your runtime logic is failing.


In [ ]:
!modal shell lesson3_images.py::make_markdown_report

## Step 6 – Student experiments

1) **Change only local assets** (fast iteration)
- Edit `assets/stopwords.txt` (add/remove words) and rerun.
- Because `add_local_dir(...)` is last, rebuilds are usually quicker.

2) **Change a pip package version** (forces rebuild)
- Change `jinja2==...` and rerun.
- Notice image steps re-run.

3) **Add a new build-time command**
- Add `run_commands("wc -l /data/book.txt")` and rerun.

4) **Return a different data type**
- Return a `dict` with stats + top words.
- Discuss: complex objects may require local deps to deserialize.

5) **Add a container-only dependency**
- Add a new library in `pip_install(...)` and import it *inside* a Modal function.
